In [1]:
!pip -q install faker pyspark
from google.colab import files
uploaded = files.upload()
print("Uploaded files:", list(uploaded.keys()))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 16.5 MB/s eta 0:00:00


Saving RETAILMART DATASET GENERATOR.txt to RETAILMART DATASET GENERATOR.txt
Uploaded files: ['RETAILMART DATASET GENERATOR.txt']


In [2]:
generator_file = next((f for f in uploaded if "GENERATOR" in f.upper() and f.lower().endswith(".txt")), None)
if generator_file is None:
    raise FileNotFoundError("Upload RETAILMART DATASET GENERATOR.txt first.")
generator_code = open(generator_file, "r", encoding="utf-8").read()
generator_code = "\n".join(line for line in generator_code.splitlines() if not line.strip().startswith("!pip"))
exec(compile(generator_code, generator_file, "exec"))

Generating 1,000,000 transaction records...
Generating transactions...
  Generated 100,000 transactions...
  Generated 200,000 transactions...
  Generated 300,000 transactions...
  Generated 400,000 transactions...
  Generated 500,000 transactions...
  Generated 600,000 transactions...
  Generated 700,000 transactions...
  Generated 800,000 transactions...
  Generated 900,000 transactions...
  Generated 1,000,000 transactions...
✅ Generated 1,000,000 transaction records

🔧 Introducing data quality issues...
  - Adding null values...
  - Adding blank strings...
  - Adding duplicate transaction_ids...
  - Adding invalid date formats...
  - Adding negative values...
  - Adding out-of-range values...
  - Adding inconsistent categorical values...

💾 Saving transactions to CSV...
✅ Saved: retailmart_transactions.csv
   - Rows: 1,000,000
   - Columns: 15
   - File size: 571.89 MB

Generating 250,000 customer feedback records...
Generating feedback records...
  Generated 50,000 feedback record

## B. Load data and inspect schema / quality

In [3]:
from google.colab import drive
import os, json, warnings
warnings.filterwarnings("ignore")
# If you uploaded files directly to the Colab session, they should be in /content.
# For persistent storage, mount Drive and set DATA_DIR to your project folder.
DATA_DIR = "/content"
TXN_PATH = os.path.join(DATA_DIR, "retailmart_transactions.csv")
FEEDBACK_PATH = os.path.join(DATA_DIR, "customer_feedback.json")
print("Transaction file exists:", os.path.exists(TXN_PATH))
print("Feedback file exists:", os.path.exists(FEEDBACK_PATH))

import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("RetailMartProject").getOrCreate()
raw_spark = spark.read.option("header", True).option("inferSchema", True).csv(TXN_PATH)
raw_spark.printSchema()
print("Rows:", raw_spark.count())
raw_spark.show(5, truncate=False)

Transaction file exists: True
Feedback file exists: True
root
 |-- transaction_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- transaction_date: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- store_location: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- customer_age: double (nullable = true)
 |-- customer_gender: string (nullable = true)
 |-- loyalty_years: integer (nullable = true)
 |-- num_visits: integer (nullable = true)
 |-- churn: string (nullable = true)

Rows: 1000000
+--------------+------------------------------------+-------------------+----------------+-------------+--------+----------+------------+--------------+--------------+------------+---------------+-------------+----------+-----+
|transaction_id|customer_id   

In [4]:
# Quality profile: nulls, blank strings, distinct IDs, duplicates, invalid ranges
from pyspark.sql.types import StringType, NumericType
schema = raw_spark.schema
quality_rows = []
n = raw_spark.count()
for field in schema.fields:
    c = field.name
    nulls = raw_spark.filter(F.col(c).isNull()).count()
    blanks = raw_spark.filter(F.trim(F.col(c).cast("string")) == "").count() if isinstance(field.dataType, StringType) else 0
    quality_rows.append((c, str(field.dataType), nulls, blanks, round(100*nulls/n, 3)))
quality_df = spark.createDataFrame(quality_rows, ["column","data_type","null_count","blank_count","null_percent"])
quality_df.show(truncate=False)

print("Duplicate transaction_id rows:")
raw_spark.groupBy("transaction_id").count().filter(F.col("count") > 1).show(10, truncate=False)

+----------------+-------------+----------+-----------+------------+
|column          |data_type    |null_count|blank_count|null_percent|
+----------------+-------------+----------+-----------+------------+
|transaction_id  |StringType() |0         |0          |0.0         |
|customer_id     |StringType() |5087      |0          |0.509       |
|transaction_date|StringType() |0         |0          |0.0         |
|product_category|StringType() |0         |0          |0.0         |
|product_name    |StringType() |0         |0          |0.0         |
|quantity        |IntegerType()|0         |0          |0.0         |
|unit_price      |DoubleType() |0         |0          |0.0         |
|total_amount    |DoubleType() |9845      |0          |0.985       |
|store_location  |StringType() |4950      |0          |0.495       |
|payment_method  |StringType() |20103     |0          |2.01        |
|customer_age    |DoubleType() |29756     |0          |2.976       |
|customer_gender |StringType() |48

In [12]:
from pyspark.sql.functions import col, when, trim, lower, initcap, try_to_timestamp, coalesce, regexp_replace, lit
from pyspark.sql.types import StringType

df = raw_spark
# Trim string fields and convert empty strings to null
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, when(trim(col(field.name)) == "", None).otherwise(trim(col(field.name))))

# Parse mixed date formats safely using try_to_timestamp and lit
df = df.withColumn(
    "transaction_ts",
    coalesce(
        try_to_timestamp(col("transaction_date"), lit("yyyy-MM-dd HH:mm:ss")),
        try_to_timestamp(col("transaction_date"), lit("yyyy/MM/dd HH:mm:ss")),
        try_to_timestamp(col("transaction_date"), lit("dd-MM-yyyy HH:mm:ss")),
        try_to_timestamp(col("transaction_date"), lit("MM/dd/yyyy")),
        try_to_timestamp(col("transaction_date"), lit("yyyy-MM-dd")),
        try_to_timestamp(col("transaction_date"), lit("yyyy-MM-dd'T'HH:mm:ss")),
        try_to_timestamp(col("transaction_date"), lit("yyyy-MM-dd HH:mm"))
    )
)
# Invalid numeric values -> null
df = df.withColumn("customer_age", when((col("customer_age") < 18) | (col("customer_age") > 120), None).otherwise(col("customer_age")))
df = df.withColumn("num_visits", when((col("num_visits") < 1) | (col("num_visits") > 100), None).otherwise(col("num_visits")))
df = df.withColumn("total_amount", when(col("total_amount") < 0, None).otherwise(col("total_amount")))
df = df.withColumn("unit_price", when(col("unit_price") < 0, None).otherwise(col("unit_price")))
df = df.withColumn("quantity", when(col("quantity") < 1, None).otherwise(col("quantity")))

# Normalize categories
df = df.withColumn("customer_gender",
    when(lower(col("customer_gender")).isin("m","male"), "Male")
    .when(lower(col("customer_gender")).isin("f","female"), "Female")
    .when(lower(col("customer_gender")).isin("non-binary","non binary"), "Non-Binary")
    .when(lower(col("customer_gender")).isin("other","x"), "Other")
    .otherwise(col("customer_gender")))
df = df.withColumn("payment_method",
    when(lower(col("payment_method")) == "credit card", "Credit Card")
    .when(lower(col("payment_method")) == "debit card", "Debit Card")
    .when(lower(col("payment_method")) == "mobile pay", "Mobile Pay")
    .when(lower(col("payment_method")) == "gift card", "Gift Card")
    .when(lower(col("payment_method")) == "cash", "Cash")
    .otherwise(col("payment_method")))
df = df.withColumn("churn",
    when(lower(col("churn")).isin("yes","y","true","1"), "Yes")
    .when(lower(col("churn")).isin("no","n","false","0"), "No")
    .otherwise(None))

# Deduplicate: transaction_id is intended to identify a transaction
df = df.dropDuplicates(["transaction_id"])

# Impute numeric values with medians; categorical values with explicit Unknown
for c in ["customer_age", "total_amount", "unit_price", "quantity", "loyalty_years", "num_visits"]:
    med = df.approxQuantile(c, [0.5], 0.01)[0]
    df = df.withColumn(c, coalesce(col(c), lit(med)))

for c in ["customer_gender", "payment_method", "store_location", "product_category", "product_name", "customer_id"]:
    df = df.withColumn(c, coalesce(col(c), lit("Unknown")))

df = df.withColumn("transaction_date", col("transaction_ts").cast("timestamp"))

# Drop rows where date or target cannot be recovered
df = df.filter(col("transaction_ts").isNotNull() & col("churn").isin("Yes","No"))

print("Raw rows:", raw_spark.count(), "Cleaned rows:", df.count())
df.printSchema()
df.show(5, truncate=False)

Raw rows: 1000000 Cleaned rows: 990078
root
 |-- transaction_id: string (nullable = true)
 |-- customer_id: string (nullable = false)
 |-- transaction_date: timestamp (nullable = true)
 |-- product_category: string (nullable = false)
 |-- product_name: string (nullable = false)
 |-- quantity: double (nullable = false)
 |-- unit_price: double (nullable = false)
 |-- total_amount: double (nullable = false)
 |-- store_location: string (nullable = false)
 |-- payment_method: string (nullable = false)
 |-- customer_age: double (nullable = false)
 |-- customer_gender: string (nullable = false)
 |-- loyalty_years: double (nullable = false)
 |-- num_visits: double (nullable = false)
 |-- churn: string (nullable = true)
 |-- transaction_ts: timestamp (nullable = true)

+--------------+------------------------------------+-------------------+----------------+---------------+--------+----------+------------+--------------+--------------+------------+---------------+-------------+----------+-----+

In [13]:
OUT_DIR = "/content/retailmart_output"
os.makedirs(OUT_DIR, exist_ok=True)
clean_csv = os.path.join(OUT_DIR, "retailmart_transactions_cleaned")
df.write.mode("overwrite").option("header", True).csv(clean_csv)
df.write.mode("overwrite").parquet(os.path.join(OUT_DIR, "retailmart_transactions_cleaned_parquet"))
print("Saved cleaned CSV folder and Parquet folder to", OUT_DIR)

# Basic before/after summary
summary = {
    "raw_rows": raw_spark.count(),
    "clean_rows": df.count(),
    "raw_duplicate_id_groups": raw_spark.groupBy("transaction_id").count().filter(F.col("count") > 1).count(),
    "clean_duplicate_id_groups": df.groupBy("transaction_id").count().filter(F.col("count") > 1).count(),
    "clean_invalid_date_rows": df.filter(F.col("transaction_ts").isNull()).count(),
    "clean_missing_target_rows": df.filter(F.col("churn").isNull()).count()
}
print(summary)

Saved cleaned CSV folder and Parquet folder to /content/retailmart_output
{'raw_rows': 1000000, 'clean_rows': 990078, 'raw_duplicate_id_groups': 5000, 'clean_duplicate_id_groups': 0, 'clean_invalid_date_rows': 0, 'clean_missing_target_rows': 0}


In [20]:
from google.colab import files
import os

# Define the target local path for the single CSV file
single_csv_path = "/content/retailmart_transactions_cleaned.csv"

print("Processing and preparing the cleaned dataset for download...")

# Coalesce the Spark DataFrame into a single partition and write to a temporary folder
temp_dir = "/content/temp_single_csv"
df.coalesce(1).write.mode("overwrite").option("header", "true").csv(temp_dir)

# Find the actual part-00000... CSV file inside the output directory and rename/move it
part_file = next((f for f in os.listdir(temp_dir) if f.startswith("part-") and f.endswith(".csv")), None)

if part_file:
    os.rename(os.path.join(temp_dir, part_file), single_csv_path)
    print(f"Cleaned data successfully consolidated into {single_csv_path}")

    # Trigger browser download
    print("Starting download to your desktop...")
    files.download(single_csv_path)
else:
    print("Error: Could not extract the unified CSV file from Spark's export.")

Processing and preparing the cleaned dataset for download...
Cleaned data successfully consolidated into /content/retailmart_transactions_cleaned.csv
Starting download to your desktop...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>